# From linear regression to AR(p)

```
Authors: Thomas Moreau
         Mathurin Massias
         Alexandre Gramfort
```

The purpose of this notebook is to show a simple way to go from linear models
for i.i.d. data with scikit-learn to **time series forecasting**.

The idea is to use a **temporal embedding** of the delayed time series
with a **Hankel matrix**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import hankel

p = 3
x = np.arange(1, 7)
x

In [ ]:
X = hankel(x, r=np.zeros(p))[:-p]
X

We have our features $X$, now we need the target variable $y$:

*Note:* We work so that the number of samples is kept constant.
It's a hard constraint of scikit-learn that any transformation
to data keeps the number of samples fixed.

In [ ]:
y = x[p:]
y

Here, we frame the forecasting problem as a regression problem: we want to predict $x_t$ from the past $p$ values $x_{t-1}, \ldots, x_{t-p}$.

Each sample is a window of size $p$ of the time series, and the target is the next value of the series. Increasing the horizon of the forecast is possible would require to predict $x_t, \ldots, x_{t+h}$ from the past values.

In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X, y)

In [ ]:
def forecast(model, x, n_samples):
    """Autoregressive (recurrent) multi-step forecast.

    Why can't we just call model.predict(X_test) directly?
    Because each predicted value feeds back as input for the next step.
    sklearn's predict() is stateless — it doesn't know about the previous
    predictions, so we need this explicit loop.

    return y_pred[order:]

    Note: statsmodels AutoReg has this built in via .predict() / .forecast().
     y_pred[i + order] = model.predict(y_pred[i:i + order][None, :]).item()

    """        # slide the window: use the last `order` predicted values as features

    order = model.coef_.size  # p: number of past time steps used as features    for i in range(n_samples):

    y_pred = np.empty(n_samples + order)  # the prediction
    # We initialize the prediction with the last "context", and then loop
    # to predict one step at a time, shifting the window forward at each step.
    y_pred[:order] = x[-order:]
    for i in range(n_samples):  # now predict successively one sample at a time
        y_pred[i + order] = model.predict(y_pred[i:i + order][None, :]).item()
    # Return only the predicted values, not the initial context.
    return y_pred[order:]

In [ ]:
x_forecast = forecast(model, x, 10)
x_forecast

<div class="alert alert-success">
    <b>EXERCISE</b>:
     <ul>
      <li>Take $x_t = \sin(t/3)$ and see with which order $p$ you can forecast such a time series. Code it using the <code>forecast()</code> function above and plot the result.</li>
      <li>Compare your result with <a href="https://www.statsmodels.org/stable/generated/statsmodels.tsa.ar_model.AutoReg.html"><code>statsmodels.tsa.ar_model.AutoReg</code></a> — the off-the-shelf AR implementation that handles multi-step forecasting and lag selection for you.</li>
    </ul>

    <b>Bonus:</b> how does the required order change for $x_t = \sin(t / 3) + \cos(t/5)$?

</div>

Solution is in: `solutions/linear-forecast-1_AR-sinusoid.py`

In [ ]:
t = np.arange(100) / 3

f = lambda t: np.sin(t)
x_past = f(t[:50])


p = 5
X = hankel(x_past, r=np.zeros(p))[:-p]
y = x_past[p:]

model = LinearRegression().fit(X, y)
x_pred = forecast(model, x_past, 50)

plt.plot(t[:50], x_past)
plt.plot(t[50:], f(t[50:]))
plt.plot(t[50:], x_pred)

## Same thing with aeon

`aeon` has an experimental **`RegressionForecaster`** that automates exactly what we built by hand:

- Builds the sliding-window (Hankel) feature matrix internally.
- Accepts **any sklearn-compatible regressor** as a plug-in.
- Handles the recurrent prediction loop so you don't write `forecast()` yourself.

The interface mirrors sklearn: `fit(y)` then `predict(y)` (or `forecast(y)` to fit+predict in one call).

In [ ]:
from aeon.forecasting import RegressionForecaster
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

# Same sine-wave dataset as above
t = np.arange(200) / 3
x_all = f(t) + 0.1 * np.random.randn(200)
x_train, x_test = x_all[:100], x_all[100:]

# window=p plays the role of the AR order.
# horizon=1 means we predict one step ahead (default).
# Any sklearn regressor can be plugged in, here we use LinearRegression.
forecaster = RegressionForecaster(window=10, regressor=RandomForestRegressor(), horizon=1)

# fit: builds the Hankel matrix and trains the regressor internally
forecaster.fit(x_train)

# iterative_forecast: applies the recurrent loop automatically — no manual forecast() needed
x_pred = forecaster.iterative_forecast(x_train, 100)
print(x_pred)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t[:100],  x_train, label="Train")
ax.plot(t[100:],  x_test,  label="Test (actual)", color="orange")
ax.plot(t[100:],  x_pred,  label="aeon RegressionForecaster", linestyle="--", color="red")
ax.axvline(t[100], color="gray", linestyle="--", linewidth=0.8)
ax.set_title("AR forecasting with aeon.RegressionForecaster (window=10, Ridge)")
ax.legend()
plt.tight_layout()